# Installations and Imports

In [1]:
# !pip install pyarrow

In [3]:
import pyarrow.dataset as ds
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
import pyarrow.dataset as ds
import gc

Data Loading

In [4]:
# 1. Load Data
print("Loading datasets...")
train_path = "/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/train-001.parquet"
test_path = "/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/test.parquet"

train_df = ds.dataset(train_path, format="parquet").to_table().to_pandas()
test_df = ds.dataset(test_path, format="parquet").to_table().to_pandas()


Loading datasets...


In [14]:
train_df

,S01_F01_U01,S01_F02_U01,S01_F03_U01,S02_F01_U01,S02_F02_U01,S02_F03_U01,S01_O01,S01_O02,S01_O01_A01,S01_O02_A01,...,S03_D02_V01_A01_B06_E06_E07_LagT3,S03_D02_V01_A01_B07_E07_E08_LagT3,S03_D02_V01_A01_B08_E08_E09_LagT3,S03_D02_V01_A01_B09_E09_E10_LagT3,S03_D02_V01_A01_B10_E10_E11_LagT3,ID,TARGET,target_bin,TARGET_CLIPPED,sample_weight
0,5.486363e+06,4.516009e+06,9.703537e+05,5.348193e+06,5.368650e+06,-2.045747e+04,-0.077096,-0.000027,-0.076575,-0.000027,...,-0.000098,-0.000549,0.006702,-0.001495,-0.002258,0,-0.009812,6,-0.009812,1.0
1,3.176045e+06,3.248363e+06,-7.231744e+04,6.167449e+06,5.760467e+06,4.069821e+05,-0.032110,-0.000006,-0.031623,-0.000006,...,0.000318,0.000458,0.000478,0.001075,0.000461,1,-0.038866,1,-0.038866,1.0
2,3.807792e+06,3.700750e+06,1.070420e+05,4.426639e+06,4.048712e+06,3.779267e+05,-0.029807,-0.000009,-0.030409,-0.000009,...,-0.004595,-0.005061,0.001462,-0.004309,-0.004066,2,-0.008135,7,-0.008135,1.0
3,1.299343e+06,1.219005e+06,8.033787e+04,1.787033e+06,1.727935e+06,5.909787e+04,-0.226014,-0.000061,-0.225756,-0.000061,...,0.000814,-0.000982,-0.000837,-0.000730,-0.000550,3,-0.004339,8,-0.004339,1.0
4,2.225848e+06,2.303032e+06,-7.718352e+04,3.886344e+06,4.187514e+06,-3.011706e+05,0.018623,0.000026,0.018278,0.000026,...,0.002469,0.001992,0.002604,-0.001380,0.003646,4,-0.001989,9,-0.001989,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
661569,1.983852e+06,1.967837e+06,1.601503e+04,3.815394e+06,4.107927e+06,-2.925330e+05,0.087069,0.000013,0.086652,0.000013,...,-0.000014,0.000613,0.000364,0.000984,0.000969,672369,-0.010247,6,-0.010247,1.0
661570,3.352151e+06,3.276458e+06,7.569341e+04,5.526027e+06,5.512316e+06,1.371024e+04,-0.192949,-0.000018,-0.192846,-0.000018,...,0.000528,0.000525,0.002229,0.002114,0.001982,672370,0.031013,17,0.031013,1.0
661571,2.518586e+06,2.835550e+06,-3.169641e+05,4.885987e+06,4.789011e+06,9.697671e+04,0.013756,0.000014,0.021370,0.000015,...,0.002996,0.005339,0.003855,-0.020997,0.010003,672371,0.015522,14,0.015522,1.0
661572,6.424048e+06,4.853257e+06,1.570791e+06,9.746562e+06,8.701088e+06,1.045474e+06,-0.123788,-0.000041,-0.121047,-0.000041,...,0.001514,0.001834,0.002206,-0.000277,0.002277,672372,0.060163,19,0.060163,1.0


# Phase 0

In [7]:

def memory_and_sparsity_check(df, name="Dataset"):
    print(f"\n{'='*15} Phase 0: Cell 1 - {name} {'='*15}")
    
    # 2. Initial Memory
    start_mem = df.memory_usage().sum() / 1024**2
    print(f"Initial Memory Usage: {start_mem:.2f} MB")
    
    # 3. Downcast float64 to float32
    f64_cols = df.select_dtypes(include='float64').columns
    df[f64_cols] = df[f64_cols].astype('float32')
    end_mem = df.memory_usage().sum() / 1024**2
    print(f"Memory after float32 downcast: {end_mem:.2f} MB (Decreased by {start_mem - end_mem:.2f} MB)")
    
    # 4. Sparsity & NaN Checks
    total_rows = len(df)
    dead_columns = []
    
    for col in df.columns:
        if col in ['ID', 'TARGET']: continue
            
        nan_count = df[col].isna().sum()
        inf_count = np.isinf(df[col]).sum()
        zero_count = (df[col] == 0).sum()
        
        nan_pct = nan_count / total_rows
        zero_pct = zero_count / total_rows
        
        # If a column is essentially empty or purely zeros
        if nan_pct > 0.995 or zero_pct > 0.995:
            dead_columns.append(col)
            
    print(f"Identified {len(dead_columns)} 'Dead' Columns (>99.5% NaNs or Zeros).")
    
    # 5. Dead Rows (All features are NaN)
    feature_cols = [c for c in df.columns if c not in ['ID', 'TARGET']]
    dead_rows = df[feature_cols].isna().all(axis=1).sum()
    print(f"Identified {dead_rows} 'Dead' Rows (100% of features are NaN).")
    
    return dead_columns, dead_rows

# Run checks
dead_cols_train, dead_rows_train = memory_and_sparsity_check(train_df, "TRAIN SET")
dead_cols_test, dead_rows_test = memory_and_sparsity_check(test_df, "TEST SET")

# VERDICT: Union of dead columns to drop in Phase 1
columns_to_drop_phase_1 = list(set(dead_cols_train + dead_cols_test))
print(f"\n[VERDICT] Total unique columns to drop in Phase 1: {len(columns_to_drop_phase_1)}")

# Free up RAM
gc.collect()


=============== Phase 0: Cell 1 - TRAIN SET ===============
Initial Memory Usage: 1130.62 MB
Memory after float32 downcast: 1130.62 MB (Decreased by 0.00 MB)
Identified 0 'Dead' Columns (>99.5% NaNs or Zeros).
Identified 0 'Dead' Rows (100% of features are NaN).

=============== Phase 0: Cell 1 - TEST SET ===============
Initial Memory Usage: 699.36 MB
Memory after float32 downcast: 699.36 MB (Decreased by 0.00 MB)
Identified 0 'Dead' Columns (>99.5% NaNs or Zeros).
Identified 0 'Dead' Rows (100% of features are NaN).

[VERDICT] Total unique columns to drop in Phase 1: 0


597

In [5]:
import scipy.stats as stats

print(f"\n{'='*15} Phase 0: Cell 2 - TARGET Deep Dive {'='*15}")

# 1. Statistical Moments
target = train_df['TARGET'].dropna()
N = len(target)

mean_val = target.mean()
var_val = target.var()
skew_val = target.skew()
kurt_val = target.kurtosis()

print(f"Mean: {mean_val:.6f} | Variance: {var_val:.6f}")
print(f"Skewness: {skew_val:.4f} | Kurtosis: {kurt_val:.4f}")
if kurt_val > 3:
    print("-> High Kurtosis detected: The target has 'fat tails' (extreme market spikes). Clipping is mandatory.")

# 2. Mathematically finding the Outlier "Elbows" using Rate of Change (Gradient)
sorted_target = np.sort(target.values)
# Calculate the absolute rate of change between sorted values
gradient = np.abs(np.gradient(sorted_target))

# We define "noise" as where the rate of change is > 3 standard deviations from the mean change
grad_mean = np.mean(gradient)
grad_std = np.std(gradient)
threshold = grad_mean + (3 * grad_std)

# Find indices where the gradient explodes (the elbows)
explosions = np.where(gradient > threshold)[0]

if len(explosions) > 0:
    lower_idx = explosions[explosions < (N / 2)][-1] if len(explosions[explosions < (N / 2)]) > 0 else int(N * 0.001)
    upper_idx = explosions[explosions > (N / 2)][0] if len(explosions[explosions > (N / 2)]) > 0 else int(N * 0.999)
else:
    # Fallback if no massive mathematical explosion is found
    print("falling back for math bound")
    lower_idx, upper_idx = int(N * 0.001), int(N * 0.999)

lower_bound_val = sorted_target[lower_idx]
upper_bound_val = sorted_target[upper_idx]

print(f"\n[VERDICT] Mathematical Target Clipping Bounds:")
print(f"Lower Bound: {lower_bound_val:.6f}")
print(f"Upper Bound: {upper_bound_val:.6f}")

# 3. Optimal Binning using Sturges' Rule for equal-frequency quantiles
optimal_k = int(np.ceil(np.log2(N) + 1))
print(f"\nCalculated Optimal Bins (Sturges' Rule): {optimal_k}")

# Create the target_bins array for StratifiedKFold
# We use qcut to ensure equal-frequency (each bin has the same number of rows)
train_df['target_bin'] = pd.qcut(train_df['TARGET'], q=optimal_k, labels=False, duplicates='drop')
valid_bins_created = train_df['target_bin'].nunique()

print(f"[VERDICT] Successfully created {valid_bins_created} equal-frequency bins for Phase 4 CV.")

# Clean up memory
del sorted_target, gradient
gc.collect()


=============== Phase 0: Cell 2 - TARGET Deep Dive ===============
Mean: -0.000036 | Variance: 0.001344
Skewness: 0.1691 | Kurtosis: 48.0903
-> High Kurtosis detected: The target has 'fat tails' (extreme market spikes). Clipping is mandatory.

[VERDICT] Mathematical Target Clipping Bounds:
Lower Bound: -0.288260
Upper Bound: 0.290438

Calculated Optimal Bins (Sturges' Rule): 21
[VERDICT] Successfully created 21 equal-frequency bins for Phase 4 CV.


0

In [11]:
import scipy.stats as stats
import numpy as np
import pandas as pd

print(f"\n{'='*15} Phase 0: Cell 2 (v2) - Robust TARGET Protection {'='*15}")

target = train_df['TARGET'].dropna()
N = len(target)

# 1. Industry Standard Bounds: Median Absolute Deviation (MAD)
# Calculate true median and MAD
# 1. Mathematically Superior Bounds: Kurtosis Targeting via MAD
median_val = np.median(target)
mad_val = stats.median_abs_deviation(target)

# Dynamically find the multiplier that forces Kurtosis down to ~3.0 (Normal Distribution)
best_multiplier = 10.0
for m in np.arange(15.0, 1.0, -0.5):
    temp_clipped = np.clip(target, median_val - (m * mad_val), median_val + (m * mad_val))
    current_kurt = stats.kurtosis(temp_clipped, fisher=False) # fisher=False makes Normal = 3.0
    if current_kurt <= 3.05:
        best_multiplier = m
        break

lower_bound_val = median_val - (best_multiplier * mad_val)
upper_bound_val = median_val + (best_multiplier * mad_val)

print(f"[VERDICT] Kurtosis-Targeted MAD Bounds ({best_multiplier}x MAD):")
# ... proceed to apply to train_df['TARGET_CLIPPED']
print(f"Lower Bound: {lower_bound_val:.6f}")
print(f"Upper Bound: {upper_bound_val:.6f}")

# 2. Apply the Hard Clip to the Training Data (Action 1 for Training Robustness)
train_df['TARGET_CLIPPED'] = np.clip(train_df['TARGET'], lower_bound_val, upper_bound_val)
new_kurt = train_df['TARGET_CLIPPED'].kurtosis()
print(f"Target Kurtosis dropped from 48.09 -> {new_kurt:.4f} after clipping!")

# 3. Stateless Sample Weights (Action 2 for Training Robustness)
# Rows near the median get weight 1.0. 
# Weights decay to ~0.1 as they approach the clipping bounds.
distances = np.abs(train_df['TARGET_CLIPPED'] - median_val)
max_distance = (multiplier * mad_val)
# Decay function (1.0 down to 0.1)
train_df['sample_weight'] = 1.0 - (0.9 * (distances / max_distance))
print(f"Generated 'sample_weight' column for model training.")

# 4. Superior Binning: Freedman-Diaconis Rule
# FD Bin Width = 2 * IQR / (N^(1/3))
iqr_val = np.percentile(target, 75) - np.percentile(target, 25)
fd_bin_width = 2 * iqr_val / (N ** (1/3))

# Calculate number of bins based on the CLIPPED range to avoid millions of empty outlier bins
k_fd = int(np.ceil((upper_bound_val - lower_bound_val) / fd_bin_width))

# Cap bins at 50 if FD goes crazy, but ensure at least 10.
optimal_k = max(10, min(k_fd, 50)) 
print(f"\nCalculated Optimal Bins (Freedman-Diaconis): {optimal_k}")

# Create the equal-frequency target_bins array for StratifiedKFold using the clipped target
train_df['target_bin'] = pd.qcut(train_df['TARGET_CLIPPED'], q=optimal_k, labels=False, duplicates='drop')
valid_bins_created = train_df['target_bin'].nunique()

print(f"[VERDICT] Successfully created {valid_bins_created} robust equal-frequency bins for Phase 4 CV.")


=============== Phase 0: Cell 2 (v2) - Robust TARGET Protection ===============
[VERDICT] Kurtosis-Targeted MAD Bounds (3.5x MAD):
Lower Bound: -0.056394
Upper Bound: 0.056394
Target Kurtosis dropped from 48.09 -> -0.1656 after clipping!
Generated 'sample_weight' column for model training.

Calculated Optimal Bins (Freedman-Diaconis): 50
[VERDICT] Successfully created 46 robust equal-frequency bins for Phase 4 CV.


In [12]:
import scipy.stats as stats
from scipy.optimize import minimize_scalar
import numpy as np
import pandas as pd

print(f"\n{'='*15} Phase 0: Cell 2 (v3) - Mathematically Optimal TARGET Protection {'='*15}")

target = train_df['TARGET'].dropna()
N = len(target)

# 1. Dynamic MAD Multiplier via Root Finding
median_val = np.median(target)
mad_val = stats.median_abs_deviation(target)

# Objective: Find the multiplier 'm' that makes the clipped kurtosis exactly 3.0 (Normal distribution)
def kurtosis_objective(m):
    clipped = np.clip(target, median_val - (m * mad_val), median_val + (m * mad_val))
    # We want to minimize the squared difference between current kurtosis and target kurtosis (3.0)
    return (pd.Series(clipped).kurtosis() - 3.0)**2

print("Dynamically solving for optimal MAD multiplier to reach Kurtosis = 3.0...")
# Search for the best multiplier between 1x and 15x MAD
res = minimize_scalar(kurtosis_objective, bounds=(1, 15), method='bounded')
optimal_multiplier = res.x

lower_bound_val = median_val - (optimal_multiplier * mad_val)
upper_bound_val = median_val + (optimal_multiplier * mad_val)

print(f"[VERDICT] Dynamically Optimized Bounds ({optimal_multiplier:.2f}x MAD):")
print(f"Lower Bound: {lower_bound_val:.6f}")
print(f"Upper Bound: {upper_bound_val:.6f}")

# 2. Apply the Hard Clip to the Training Data
train_df['TARGET_CLIPPED'] = np.clip(train_df['TARGET'], lower_bound_val, upper_bound_val)
new_kurt = train_df['TARGET_CLIPPED'].kurtosis()
print(f"Target Kurtosis perfectly dropped from 48.09 -> {new_kurt:.4f}!")

# 3. Huber-Style Sample Weights
# Weight = 1.0 inside the bounds, asymptotic decay outside the bounds
distances = np.abs(train_df['TARGET'] - median_val)
threshold = optimal_multiplier * mad_val

# np.where(condition, true_value, false_value)
train_df['sample_weight'] = np.where(
    distances <= threshold, 
    1.0, 
    threshold / distances
)
print("Generated 'sample_weight' column using Huber asymptotic decay.")

# 4. ML-Safe Equal Frequency Binning (Ventiles)
optimal_k = 20 # Locked at 20 for stable CV fold population
train_df['target_bin'] = pd.qcut(train_df['TARGET_CLIPPED'], q=optimal_k, labels=False, duplicates='drop')
valid_bins_created = train_df['target_bin'].nunique()

print(f"[VERDICT] Successfully created {valid_bins_created} robust equal-frequency bins for Phase 4 CV.")


=============== Phase 0: Cell 2 (v3) - Mathematically Optimal TARGET Protection ===============
Dynamically solving for optimal MAD multiplier to reach Kurtosis = 3.0...
[VERDICT] Dynamically Optimized Bounds (8.28x MAD):
Lower Bound: -0.133407
Upper Bound: 0.133407
Target Kurtosis perfectly dropped from 48.09 -> 3.0000!
Generated 'sample_weight' column using Huber asymptotic decay.
[VERDICT] Successfully created 20 robust equal-frequency bins for Phase 4 CV.


In [18]:
train_df['sample_weight'].nunique()

5190

In [19]:
import pandas as pd
import numpy as np
import itertools
import json

print(f"\n{'='*15} Phase 0: Cell 3 - Market Physics & De-Anonymization {'='*15}")

# 1. Isolate Base Features vs Lag Features
# Exclude metadata, target, and the Phase 2 generated columns
non_features = ['ID', 'TARGET', 'TARGET_CLIPPED', 'sample_weight', 'target_bin']
all_features = [c for c in train_df.columns if c not in non_features]

base_features = [c for c in all_features if 'Lag' not in c]
lag_features = [c for c in all_features if 'Lag' in c]

print(f"Total Predictive Features: {len(all_features)}")
print(f"Identified Base Features: {len(base_features)} (Including Price & SO3_T)")

# 2. Prefix Lineage Parsing (Family Grouping)
feature_families = {}
for col in base_features:
    if col in ['Price', 'SO3_T']:
        family = col
    else:
        # Extract the first block (e.g., 'S01' from 'S01_F01_U01')
        family = col.split('_')[0] 
        
    if family not in feature_families:
        feature_families[family] = []
    feature_families[family].append(col)

print("\n[VERDICT] Feature Families Identified:")
for fam, cols in feature_families.items():
    print(f"  - Family '{fam}': {len(cols)} base features")

# 3. Non-Negative Hunt (Volumes, Spreads, Volatility)
non_negative_cols = []
for col in base_features:
    if train_df[col].min() >= -1e-5: # Account for float32 precision
        non_negative_cols.append(col)

print(f"\n[VERDICT] Identified {len(non_negative_cols)} Non-Negative Base Features.")
print("These are strictly physical bounds (Volumes, Absolute Spreads, or Counts).")

# 4. Constraint Hunting (Bid/Ask or High/Low Structural Pairs)
print("\nHunting for Structural Pairs (A >= B)...")
corr_matrix = train_df[base_features].corr().abs()
structural_pairs = []

# Only check highly correlated pairs to save time and avoid spurious random matches
high_corr_pairs = np.where((corr_matrix > 0.85) & (corr_matrix < 0.999))
checked = set()

for idx1, idx2 in zip(high_corr_pairs[0], high_corr_pairs[1]):
    if idx1 >= idx2: continue # Avoid duplicates
    
    col_A = base_features[idx1]
    col_B = base_features[idx2]
    
    diff = train_df[col_A] - train_df[col_B]
    
    # If Col_A is always >= Col_B
    if diff.min() >= -1e-5:
        structural_pairs.append((col_A, col_B, "A >= B"))
    # If Col_B is always >= Col_A
    elif diff.max() <= 1e-5:
        structural_pairs.append((col_B, col_A, "B >= A"))

print(f"[VERDICT] Found {len(structural_pairs)} strict structural pairs (e.g., Ask >= Bid).")
if len(structural_pairs) > 0:
    for pair in structural_pairs[:5]: # Print first 5
        print(f"  - {pair[0]} is always >= {pair[1]}")

# 5. Market Regime & Volatility Isolation
print("\nTesting Volatility Proxies against Target Variance...")
train_df['abs_target'] = train_df['TARGET'].abs()

# Proxy A: The published standalone covariate
so3_corr = train_df['SO3_T'].corr(train_df['abs_target'])

# Proxy B: Row-level lag turbulence on Price
# Sum of absolute price changes over the lags = total path distance
price_lags = [c for c in lag_features if 'Price' in c]
if len(price_lags) == 3:
    train_df['Price_Path_Length'] = train_df[price_lags].abs().sum(axis=1)
    path_corr = train_df['Price_Path_Length'].corr(train_df['abs_target'])
else:
    path_corr = 0

print(f"  - SO3_T correlation with Market Volatility: {so3_corr:.4f}")
print(f"  - Price_Path_Length correlation with Market Volatility: {path_corr:.4f}")

if abs(so3_corr) > abs(path_corr):
    best_proxy = 'SO3_T'
else:
    best_proxy = 'Price_Path_Length'
print(f"[VERDICT] Best Regime/Volatility Proxy selected: {best_proxy}")

# 6. Mean-Reversion vs Momentum Profiling
if 'Price_LagT1' in train_df.columns:
    # Correlation between recent price change and FUTURE price change (Target)
    micro_momentum = train_df['Price_LagT1'].corr(train_df['TARGET'])
    print(f"\n[VERDICT] Micro-Structural Signature:")
    if micro_momentum < 0:
        print(f"  - Mean-Reverting (Corr: {micro_momentum:.4f}). High negative correlation.")
        print("  - Strategy: Phase 2 should focus on oscillators and reversal features.")
    else:
        print(f"  - Momentum (Corr: {micro_momentum:.4f}). Positive correlation.")
        print("  - Strategy: Phase 2 should focus on trend continuation and acceleration.")

# Clean up
train_df.drop(columns=['abs_target', 'Price_Path_Length'], errors='ignore', inplace=True)


=============== Phase 0: Cell 3 - Market Physics & De-Anonymization ===============
Total Predictive Features: 445
Identified Base Features: 112 (Including Price & SO3_T)

[VERDICT] Feature Families Identified:
  - Family 'S01': 7 base features
  - Family 'S02': 7 base features
  - Family 'S04': 2 base features
  - Family 'S05': 1 base features
  - Family 'SO3_T': 1 base features
  - Family 'Price': 1 base features
  - Family 'S03': 93 base features

[VERDICT] Identified 58 Non-Negative Base Features.
These are strictly physical bounds (Volumes, Absolute Spreads, or Counts).

Hunting for Structural Pairs (A >= B)...
[VERDICT] Found 32 strict structural pairs (e.g., Ask >= Bid).
  - S03_V06_V15_W02 is always >= SO3_T
  - S03_V06_V15_O04 is always >= S03_V06_V15_O03
  - S03_V06_V15_O04 is always >= S03_P01_D04
  - S03_V06_V15_O04 is always >= S03_P01_D01_S01
  - S03_V06_V15_O04 is always >= S03_P01_D01_S02

Testing Volatility Proxies against Target Variance...
  - SO3_T correlation with

In [20]:
import pandas as pd
import numpy as np
import scipy.stats as stats

print(f"\n{'='*15} Phase 0: Cell 3 (v2) - Deep Market Physics {'='*15}")

# Setup
non_features = ['ID', 'TARGET', 'TARGET_CLIPPED', 'sample_weight', 'target_bin']
all_features = [c for c in train_df.columns if c not in non_features]
base_features = [c for c in all_features if 'Lag' not in c]
lag_features = [c for c in all_features if 'Lag' in c]

train_df['abs_target'] = train_df['TARGET'].abs()

# =====================================================================
# 1. The Composite "Smart Volatility" Proxy
# =====================================================================
print("\nExtracting Composite Volatility Proxy...")
# We check which absolute lag features best predict the absolute target (variance)
abs_lags = train_df[lag_features].abs()
vol_corrs = abs_lags.corrwith(train_df['abs_target']).sort_values(ascending=False)

top_5_vol_features = vol_corrs.head(5)
print("[VERDICT] Top 5 Variance Predictors:")
for feat, corr in top_5_vol_features.items():
    print(f"  - {feat}: {corr:.4f}")

# Build the composite proxy (weighted by their correlation)
train_df['Smart_Vol_Proxy'] = 0
for feat, corr in top_5_vol_features.items():
    # Normalize the feature to 0-1 so scale doesn't dominate, then weight by correlation
    feat_min, feat_max = abs_lags[feat].min(), abs_lags[feat].max()
    normalized_feat = (abs_lags[feat] - feat_min) / (feat_max - feat_min + 1e-8)
    train_df['Smart_Vol_Proxy'] += normalized_feat * corr

final_vol_corr = train_df['Smart_Vol_Proxy'].corr(train_df['abs_target'])
print(f"[VERDICT] 'Smart_Vol_Proxy' created. Correlation with Market Volatility: {final_vol_corr:.4f}")

# =====================================================================
# 2. Regime-Conditioned Micro-Structure (Dynamic Weighted Correlation)
# =====================================================================
print("\nAnalyzing Regime-Conditioned Micro-Structure...")
# Bin the market into 3 regimes based on our new Smart Vol Proxy
train_df['Vol_Regime'] = pd.qcut(train_df['Smart_Vol_Proxy'], q=[0, 0.33, 0.66, 1.0], labels=['Quiet', 'Normal', 'Volatile'])

def weighted_correlation(x, y, w):
    """Calculates correlation weighted by the absolute size of the target."""
    sum_w = np.sum(w)
    mean_x = np.sum(x * w) / sum_w
    mean_y = np.sum(y * w) / sum_w
    cov_xy = np.sum(w * (x - mean_x) * (y - mean_y)) / sum_w
    cov_xx = np.sum(w * (x - mean_x)**2) / sum_w
    cov_yy = np.sum(w * (y - mean_y)**2) / sum_w
    return cov_xy / np.sqrt(cov_xx * cov_yy + 1e-8)

if 'Price_LagT1' in train_df.columns:
    print("[VERDICT] Weighted Correlation (Price_LagT1 vs TARGET) by Regime:")
    for regime in ['Quiet', 'Normal', 'Volatile']:
        mask = train_df['Vol_Regime'] == regime
        
        # Standard Pearson
        pearson = train_df.loc[mask, 'Price_LagT1'].corr(train_df.loc[mask, 'TARGET'])
        
        # Weighted (giving more importance to rows where TARGET magnitude is larger within the regime)
        weights = train_df.loc[mask, 'abs_target'] + 1e-5 # prevent zero weights
        weighted_corr = weighted_correlation(
            train_df.loc[mask, 'Price_LagT1'].values, 
            train_df.loc[mask, 'TARGET'].values, 
            weights.values
        )
        print(f"  - {regime} Market -> Pearson: {pearson:.4f} | Weighted: {weighted_corr:.4f}")

# =====================================================================
# 3. Sub-Family Clustering for 'S03'
# =====================================================================
print("\nUnpacking the massive 'S03' Family...")
s03_features = [c for c in base_features if c.startswith('S03_')]
s03_subfamilies = {}

for col in s03_features:
    # Extract the first two blocks (e.g., 'S03_V06')
    parts = col.split('_')
    if len(parts) >= 2:
        sub = f"{parts[0]}_{parts[1]}"
        s03_subfamilies[sub] = s03_subfamilies.get(sub, 0) + 1

print(f"[VERDICT] 'S03' is composed of {len(s03_subfamilies)} distinct sub-families:")
# Sort by count
for sub, count in sorted(s03_subfamilies.items(), key=lambda item: item[1], reverse=True)[:5]:
    print(f"  - {sub}: {count} features")
if len(s03_subfamilies) > 5:
    print("  - ...and others.")

# =====================================================================
# 4. Alpha Factor Baseline (Spearman Rank IC)
# =====================================================================
print("\nCalculating Spearman Rank IC for Base Features...")
# Spearman evaluates the monotonic relationship, ignoring outlier distortions
ic_scores = []
for col in base_features:
    # Drop NaNs just for the calculation
    valid_mask = ~train_df[col].isna() & ~train_df['TARGET'].isna()
    if valid_mask.sum() > 100:
        # Spearman R
        corr, pval = stats.spearmanr(train_df.loc[valid_mask, col], train_df.loc[valid_mask, 'TARGET'])
        ic_scores.append({'Feature': col, 'Rank_IC': corr, 'Abs_IC': abs(corr)})

ic_df = pd.DataFrame(ic_scores).sort_values(by='Abs_IC', ascending=False)
print("[VERDICT] Top 5 Strongest Raw Alpha Signals (Rank IC):")
print(ic_df.head(5).to_string(index=False))

# Clean up temporary columns
train_df.drop(columns=['abs_target', 'Vol_Regime'], inplace=True)
# Keep Smart_Vol_Proxy! We need it for Phase 1 scaling.
del abs_lags, ic_df
import gc; gc.collect()


=============== Phase 0: Cell 3 (v2) - Deep Market Physics ===============

Extracting Composite Volatility Proxy...
[VERDICT] Top 5 Variance Predictors:
  - Price_LagT3: 0.3530
  - Price_LagT2: 0.3467
  - S03_P01_D01_S02_LagT2: 0.3332
  - Price_LagT1: 0.3272
  - S03_P01_D01_S02_LagT3: 0.3270
[VERDICT] 'Smart_Vol_Proxy' created. Correlation with Market Volatility: 0.4266

Analyzing Regime-Conditioned Micro-Structure...
[VERDICT] Weighted Correlation (Price_LagT1 vs TARGET) by Regime:
  - Quiet Market -> Pearson: -0.0040 | Weighted: 0.0014
  - Normal Market -> Pearson: -0.0063 | Weighted: 0.0205
  - Volatile Market -> Pearson: -0.0079 | Weighted: -0.0041

Unpacking the massive 'S03' Family...
[VERDICT] 'S03' is composed of 18 distinct sub-families:
  - S03_D02: 22 features
  - S03_A07: 10 features
  - S03_P01: 9 features
  - S03_A02: 7 features
  - S03_V06: 6 features
  - ...and others.

Calculating Spearman Rank IC for Base Features...
[VERDICT] Top 5 Strongest Raw Alpha Signals (Rank

0

In [21]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from sklearn.feature_selection import mutual_info_regression

print(f"\n{'='*15} Phase 0: Cell 3.5 - Institutional Quant Metrics {'='*15}")

# 1. Non-Linear Signal Detection (Mutual Information)
print("\nCalculating Non-Linear Information (Mutual Information Regression)...")
# MI is extremely computationally heavy. We randomly sample 20,000 rows to estimate it.
sample_idx = np.random.choice(train_df.index, size=20000, replace=False)
sample_X = train_df.loc[sample_idx, base_features].fillna(0) # MI doesn't handle NaNs well
sample_y = train_df.loc[sample_idx, 'TARGET']

# Calculate MI
mi_scores = mutual_info_regression(sample_X, sample_y, random_state=42)
mi_df = pd.Series(mi_scores, index=base_features).sort_values(ascending=False)

print("[VERDICT] Top 5 True Non-Linear Signals (MI Scores):")
print(mi_df.head(5).to_string())
print("-> ACTION: In Phase 2, prioritize these features for advanced interactions (e.g., rolling ratios, multiplication).")

# 2. Family Multicollinearity (Condition Number)
print("\nChecking Feature Family Redundancy (Matrix Condition Numbers)...")
# We will check the S03_D02 family you identified which has 22 features
s03_d02_cols = [c for c in base_features if c.startswith('S03_D02')]

# Drop NaNs for matrix math
clean_matrix = train_df[s03_d02_cols].dropna().sample(n=5000) 
# Standardize before condition number
clean_matrix = (clean_matrix - clean_matrix.mean()) / (clean_matrix.std() + 1e-8)

# Calculate singular values to find the condition number
_, s, _ = np.linalg.svd(clean_matrix, full_matrices=False)
cond_number = s[0] / s[-1]

print(f"[VERDICT] Condition Number for Family 'S03_D02': {cond_number:.2f}")
if cond_number > 30:
    print("-> ACTION: Severe multicollinearity detected. In Phase 1, we MUST apply PCA to compress 'S03_D02' to 3-5 components, otherwise tree models will split on redundant noise.")
else:
    print("-> ACTION: Features are orthogonal enough. Keep as is.")

# 3. Target Tail Physics (Hill Estimator for Huber Delta)
print("\nCalculating Target Tail Index for Custom Loss Function...")
target_vals = train_df['TARGET'].dropna().values
target_vals = np.abs(target_vals - np.median(target_vals))
target_vals = np.sort(target_vals)

# Focus on the top 5% of extreme values (the right tail)
k = int(len(target_vals) * 0.05)
extreme_tail = target_vals[-k:]
threshold = target_vals[-k-1]

# Hill Estimator Formula
hill_estimator = (1 / k) * np.sum(np.log(extreme_tail / threshold))
tail_index = 1 / hill_estimator

print(f"[VERDICT] Tail Index (Alpha): {tail_index:.4f}")
print(f"-> ACTION: For Phase 6 Custom Huber Loss, set delta at exactly the threshold where tails begin to dominate.")
print(f"-> EXACT HUBER DELTA TO USE: {threshold:.6f}")

import gc; gc.collect()


=============== Phase 0: Cell 3.5 - Institutional Quant Metrics ===============

Calculating Non-Linear Information (Mutual Information Regression)...
[VERDICT] Top 5 True Non-Linear Signals (MI Scores):
S03_V03_T06            0.095641
S03_V06_V01            0.094091
S03_V06_V15_V01        0.093916
S03_V03_T04            0.093797
S03_A07_V01_V16_V06    0.092208
-> ACTION: In Phase 2, prioritize these features for advanced interactions (e.g., rolling ratios, multiplication).

Checking Feature Family Redundancy (Matrix Condition Numbers)...
[VERDICT] Condition Number for Family 'S03_D02': 69.40
-> ACTION: Severe multicollinearity detected. In Phase 1, we MUST apply PCA to compress 'S03_D02' to 3-5 components, otherwise tree models will split on redundant noise.

Calculating Target Tail Index for Custom Loss Function...
[VERDICT] Tail Index (Alpha): 2.7884
-> ACTION: For Phase 6 Custom Huber Loss, set delta at exactly the threshold where tails begin to dominate.
-> EXACT HUBER DELTA TO U

0